In [7]:
from transformers import M2M100ForConditionalGeneration, M2M100Tokenizer
from sentence_transformers import SentenceTransformer, util
from datasets import load_dataset
import pandas as pd
import torch
import sacrebleu
from tqdm import tqdm

In [8]:
# Load model and tokenizer
model_name = "facebook/m2m100_418M"
tokenizer = M2M100Tokenizer.from_pretrained(model_name)
model = M2M100ForConditionalGeneration.from_pretrained(model_name)

In [9]:
# Translation function using M2M100
def m2m_translate(texts, src_lang, tgt_lang):
    tokenizer.src_lang = src_lang
    encoded = tokenizer(texts, return_tensors="pt", padding=True, truncation=True)
    forced_bos_token_id = tokenizer.get_lang_id(tgt_lang)
    with torch.no_grad():
        generated_tokens = model.generate(**encoded, forced_bos_token_id=forced_bos_token_id)
    return [tokenizer.decode(t, skip_special_tokens=True) for t in generated_tokens]

In [10]:
# Load SentenceTransformer model for evaluation
sim_model = SentenceTransformer('sentence-transformers/distiluse-base-multilingual-cased-v2')

In [12]:
# BLEU, chrF, cosine
def evaluate(original, back_translated):
    bleu = sacrebleu.corpus_bleu([back_translated], [[original]]).score
    chrf = sacrebleu.corpus_chrf([back_translated], [[original]]).score
    cosine = semantic_similarity(original, back_translated)
    return round(bleu, 2), round(chrf, 2), round(cosine, 4)

In [13]:
# Load Welsh CEFR data
welsh_data = load_dataset("UniversalCEFR/learn_welsh_cy")["train"]
welsh_a1_a2 = welsh_data.filter(lambda example: example["cefr_level"] in ["A1", "A2"])
df = welsh_a1_a2.to_pandas()[["text", "cefr_level"]].dropna().reset_index(drop=True)

In [14]:
# Translation loop
records = []

for _, row in tqdm(df.iterrows(), total=len(df), desc="Back-translating (cy-fr-cy)"):
    original_cy = row["text"]
    cefr = row["cefr_level"]

    try:
        # cy → fr
        french = m2m_translate([original_cy], src_lang="cy", tgt_lang="fr")[0]

        # fr → cy
        back_cy = m2m_translate([french], src_lang="fr", tgt_lang="cy")[0]

        # Evaluate
        bleu, chrf, cosine = evaluate(original_cy, back_cy)

        records.append({
            "original_welsh": original_cy,
            "translated_french": french,
            "back_translated_welsh": back_cy,
            "cefr_level": cefr,
            "BLEU": bleu,
            "chrF": chrf,
            "cosine_similarity": cosine
        })

    except Exception as e:
        print(f"Error on: {original_cy}\n{e}")

Back-translating (cy-fr-cy): 100%|██████████| 1372/1372 [6:06:56<00:00, 16.05s/it]  


In [15]:
df_bt = pd.DataFrame(records)
# filtered_df = df_bt[df_bt["cosine_similarity"] > 0.80]
df_bt.to_csv("files/welsh_back_translation_fr_FB.csv", index=False)

In [16]:
df_bt

,original_welsh,translated_french,back_translated_welsh,cefr_level,BLEU,chrF,cosine_similarity
0,"A: Helô, Eryl dw i. Pwy dych chi?\nB: Bore da,...","A: Halo, Eryl deux i. Que pensez-vous? B: Bore...","A: Halo, Eryl 2 i. Mae'r gynnu? B: Bore da, Ce...",A1,45.77,38.84,0.7199
1,"A: O na, yr heddlu! (Stopio'r car)\nB: Hello, ...","A: O, le message! (Stop car) B: Hello, hello, ...","A: O, mae'r messenger! (Stop car) B: Hello, he...",A1,29.62,29.88,0.7158
2,"A: Bore da. Sut dych chi?\nB: Iawn, ond wedi b...","A: Bore da. Qu'est-ce que j'ai dit? B: Iawn, m...","A: Bore da. Mae'n cael ei wneud? B: Iawn, mae'...",A1,14.60,18.51,0.7383
3,"Ceri: Noswaith dda, Eryl. Sut wyt ti?\nEryl: D...","Re: Bonne nouvelle, Eryl. Qu'est-ce que vous v...",Re: Mae'r newyddion gan Eryl. Mae'r newyddion ...,A1,1.54,12.52,0.4339
4,A: Bore da.\nB: Hmff.\nA: Sut dych chi heddiw?...,A: Bore da. B: Hmff. A: Qu'est-ce que je pense...,A: Bore da. B: Hmff. A: Mae'n ddyn yn ôl? B: M...,A1,12.55,12.63,0.6844
...,...,...,...,...,...,...,...
1367,Allech chi gyrraedd yn gynnar?,Qu’est-ce qu’il y a dans le monde ?,Mae'n cael ei wneud yn y byd?,A2,6.57,13.94,0.4152
1368,Allet ti gyrraedd yn gynnar?,Est-ce que vous êtes en train de travailler ?,Mae'n cael ei ddefnyddio?,A2,8.75,10.69,0.4650
1369,Allai hi gyrraedd yn gynnar?,Est-ce qu’il y a de l’argent ?,Mae'n cael eu pen?,A2,8.75,6.04,0.4210
1370,Allen nhw gyrraedd yn gynnar?,Est-ce qu’ils sont en train d’aller ?,Mae'n cael ei wneud yn ôl?,A2,7.81,12.58,0.4397
